# Digital Twin (DART) — HTTP Colab pipeline (clinical JSON actions)

This mirrors a **judge-style** step list, but wired to **this repo**:

- **No** `generate_patient_dataset.py` — patients come from **`PatientTwin`** on each `reset`.
- **No** `environment.patient_env` — the FastAPI app is **`dtm_openenv.server.app:app`**.
- Actions are **not** `INSULIN` / `FLAG` strings. They must match **`DTMAction`**: `type`, `drug`, `dose`, `lifestyle`, … (same schema as `env/action_parser.py`).
- Observations are **partial vitals**: `week`, `hba1c`, `fasting_glucose`, `bmi`, `systolic_bp`, `egfr`, `ckd`, `cvd` (see `DigitalTwinDiabetesEnv`).

**Compare (HTTP rollouts):**
1. **Rule-based clinician** — thresholds on vitals → valid JSON actions  
2. **Small LM** — `distilgpt2` text → parsed JSON (fallback `noop`)  
3. **Large LM** (optional GPU) — stronger model, same contract  

**Honesty:** this notebook evaluates **policies per episode over the HTTP API**. Your main **in-env REINFORCE** evidence and figures are in **`training/DART_Colab_submission.ipynb`**. This notebook is for **live server + API narrative**; that file is the **single** Colab to submit for all-in-one training + plots (or use `train_reinforce_twin.py` locally).

## STEP 1 — Clone repo & install
Set `YOUR_REPO_URL` (e.g. `https://github.com/mano45sudo-lgtm/DART.git`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from typing import Optional

YOUR_REPO_URL = "https://github.com/mano45sudo-lgtm/DART.git"
CLONE_DIR = Path("/content/dart_repo")
_marker = Path("scripts") / "train_reinforce_twin.py"


def _find_dart_root(base: Path) -> Optional[Path]:
    nested = base / "DART" / _marker
    root = base / _marker
    if nested.is_file():
        return (base / "DART").resolve()
    if root.is_file():
        return base.resolve()
    return None


if not CLONE_DIR.is_dir():
    subprocess.run(["git", "clone", YOUR_REPO_URL, str(CLONE_DIR)], check=True)

REPO_ROOT = _find_dart_root(CLONE_DIR)
if REPO_ROOT is None and (CLONE_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)
    REPO_ROOT = _find_dart_root(CLONE_DIR)
if REPO_ROOT is None:
    raise FileNotFoundError("Fix CLONE_DIR or YOUR_REPO_URL; see colab_judge_pipeline STEP 1.")

os.chdir(REPO_ROOT)
os.environ["DART_REPO_ROOT"] = str(REPO_ROOT.resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements_hackathon.txt"],
    check=True,
)
print("REPO_ROOT:", REPO_ROOT)

## STEP 2 — Enable GPU
`Runtime → Change runtime type → GPU` (needed for STEP 9 large LM; optional for rule + small LM HTTP eval).

## STEP 3 — Hugging Face token & large model id
Edit the next cell: replace the HF placeholder with your **read** token in Colab (keep placeholder in git).

**Model id:** `unsloth/mistral-7b-instruct-bnb-4bit` is not a Hub repo. Use e.g. **`unsloth/mistral-7b-instruct-v0.3-bnb-4bit`** (default below) or **`unsloth/mistral-7b-instruct-v0.2-bnb-4bit`**.

In [ ]:
import os
from pathlib import Path

_HF_PLACEHOLDER = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["HF_TOKEN"] = _HF_PLACEHOLDER  # in Colab: paste your real read token here (keep placeholder in git)

os.environ["MODEL_ID"] = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
os.environ["MODEL_ID_LARGE"] = os.environ["MODEL_ID"]

from huggingface_hub import login

if os.environ["HF_TOKEN"].strip() and os.environ["HF_TOKEN"] != _HF_PLACEHOLDER:
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"])
os.chdir(REPO_ROOT)
print("MODEL_ID:", os.environ["MODEL_ID"])

## STEP 4 — “Generate patient dataset” (DART: no separate script)
Synthetic **patient instances** are created inside the twin on **`/reset`** (`PatientTwin`). Run sanity to verify imports.

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "scripts/run_sanity.py"], cwd=Path(os.environ["DART_REPO_ROOT"]), check=True)

## STEP 5 — Start Digital Twin **HTTP** environment
Correct ASGI path: **`dtm_openenv.server.app:app`** (not `environment.patient_env`).
`PYTHONPATH` must include the **DART repo root** so `env` / `dtm_openenv` import correctly.

In [ ]:
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"]).resolve()
envp = {**os.environ, "PYTHONPATH": str(REPO_ROOT)}


def _run_uvicorn():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "dtm_openenv.server.app:app",
            "--host",
            "0.0.0.0",
            "--port",
            "8000",
        ],
        cwd=str(REPO_ROOT),
        env=envp,
    )


th = threading.Thread(target=_run_uvicorn, daemon=True)
th.start()
time.sleep(6)
print("Uvicorn thread started. Health:", end=" ")
import requests

print(requests.get("http://127.0.0.1:8000/health", timeout=10).json())

## STEP 6 — Evaluation helper (matches **this** API)
`POST /reset` returns `{"observation": {"observation": {vitals...}, "info": ...}, "state": ...}`.
`POST /step` expects a **flat** JSON body: `{"type":"start","drug":"metformin",...}` (same as `DTMAction`).
`done` is a **single boolean** (terminated or truncated).

In [ ]:
import random
import time
from typing import Any, Callable, Dict, List

import requests

BASE = "http://127.0.0.1:8000"


def inner_obs(api_json: Dict[str, Any]) -> Dict[str, Any]:
    """Unpack DTMObservation.model_dump() from /reset and /step."""
    o = api_json.get("observation", {})
    if isinstance(o, dict) and "observation" in o and isinstance(o["observation"], dict):
        return o["observation"]
    if isinstance(o, dict):
        return o
    return {}


def run_episode(agent_fn: Callable[[Dict[str, Any]], Dict[str, Any]], *, n: int = 12, max_steps: int = 52) -> float:
    """Mean total reward over n HTTP episodes (new seed each reset)."""
    rewards: List[float] = []
    for _ in range(n):
        seed = random.randint(0, 10_000)
        r = requests.post(f"{BASE}/reset", json={"seed": seed}, timeout=60)
        r.raise_for_status()
        data = r.json()
        obs = inner_obs(data)
        total = 0.0
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = agent_fn(obs)
            rr = requests.post(f"{BASE}/step", json=action, timeout=60)
            rr.raise_for_status()
            out = rr.json()
            total += float(out.get("reward", 0.0))
            done = bool(out.get("done", False))
            obs = inner_obs(out)
            steps += 1
        rewards.append(total)
    return float(sum(rewards) / max(len(rewards), 1))

## STEP 7 — Baseline “doctor” (rule-based → **valid JSON actions**)
Uses vitals from the twin observation (not `obs["patient"]["glucose"]` — that shape is from the template, not DART).

In [ ]:
from typing import Any, Dict


def baseline_doctor(obs: Dict[str, Any]) -> Dict[str, Any]:
    """Simple guideline-style policy: always returns a DTMAction-compatible dict."""
    fg = float(obs.get("fasting_glucose", 120.0))
    hba1c = float(obs.get("hba1c", 6.5))
    sbp = float(obs.get("systolic_bp", 120.0))
    if fg >= 200 or hba1c >= 9.0:
        return {"type": "start", "drug": "metformin", "dose": 1.0, "lifestyle": 0.65}
    if fg >= 160 or hba1c >= 8.0:
        return {"type": "start", "drug": "metformin", "dose": 0.9, "lifestyle": 0.7}
    if fg >= 140 or hba1c >= 7.2:
        return {"type": "start", "drug": "metformin", "dose": 0.8, "lifestyle": 0.72}
    if sbp >= 140:
        return {"type": "start", "drug": "metformin", "dose": 0.6, "lifestyle": 0.75}
    return {"type": "noop"}

## STEP 8 — Small LM (DistilGPT2 → JSON action, not `INSULIN` strings)
We reuse **`env.action_parser.safe_action`** on model text so invalid generations fall back to `noop` safely.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"])
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from env.action_parser import safe_action
from training.llm_reinforce import obs_to_prompt
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_tok = AutoTokenizer.from_pretrained("distilgpt2", use_fast=True)
if _tok.pad_token is None:
    _tok.pad_token = _tok.eos_token
_lm = AutoModelForCausalLM.from_pretrained("distilgpt2").to(_device)
_lm.eval()


def small_lm_agent(obs: dict) -> dict:
    prompt = obs_to_prompt(obs)
    enc = _tok(prompt, return_tensors="pt").to(_device)
    with torch.no_grad():
        gen = _lm.generate(
            **enc,
            max_new_tokens=48,
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            pad_token_id=_tok.pad_token_id,
        )
    q = int(enc["input_ids"].shape[1])
    text = _tok.decode(gen[0, q:], skip_special_tokens=True).strip()
    action, _info = safe_action(text)
    return action

print("Small LM ready on", _device)

## STEP 9 — Large LM (optional; 4-bit if CUDA)
Same contract as STEP 8. If this fails OOM, skip and compare **rule vs small** only, or use a smaller `MODEL_ID_LARGE`.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"])
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from env.action_parser import safe_action
from training.llm_reinforce import obs_to_prompt

mid = os.environ["MODEL_ID_LARGE"]
_tok_l = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
if _tok_l.pad_token is None:
    _tok_l.pad_token = _tok_l.eos_token

if torch.cuda.is_available():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    _lm_l = AutoModelForCausalLM.from_pretrained(
        mid,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    _dev_l = torch.device(next(_lm_l.parameters()).device)
else:
    _lm_l = AutoModelForCausalLM.from_pretrained(mid, trust_remote_code=True).to("cpu")
    _dev_l = torch.device("cpu")

_lm_l.eval()


def large_lm_agent(obs: dict) -> dict:
    prompt = obs_to_prompt(obs)
    enc = _tok_l(prompt, return_tensors="pt").to(_dev_l)
    with torch.no_grad():
        gen = _lm_l.generate(
            **enc,
            max_new_tokens=64,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=_tok_l.pad_token_id,
        )
    q = int(enc["input_ids"].shape[1])
    text = _tok_l.decode(gen[0, q:], skip_special_tokens=True).strip()
    action, _info = safe_action(text)
    return action

print("Large LM ready:", mid, "device", _dev_l)

## STEP 10 — Run evaluation (mean episode return)
Reward is the **twin rubric** (`reward/rubric.py`), not a hand-waved “outcome score”.

In [ ]:
baseline_score = run_episode(baseline_doctor, n=10)
small_score = run_episode(small_lm_agent, n=10)
large_score = run_episode(large_lm_agent, n=6)  # fewer episodes if slow

print("Rule-based doctor (mean return):", round(baseline_score, 3))
print("Small LM (mean return):        ", round(small_score, 3))
print("Large LM (mean return):      ", round(large_score, 3))

## STEP 11 — Two bar charts (judge-style)
Y-axis: **mean cumulative reward** over HTTP episodes (same `n` as STEP 10 if you want strict comparability).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(["Rule doctor", "Small LM"], [baseline_score, small_score], color=["#64748b", "#2563eb"])
plt.ylabel("Mean episode return (twin rubric)")
plt.title("Baseline vs small LM (HTTP rollouts)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(["Rule doctor", "Large LM"], [baseline_score, large_score], color=["#64748b", "#ea580c"])
plt.ylabel("Mean episode return (twin rubric)")
plt.title("Baseline vs large LM (HTTP rollouts)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## STEP 12 — Bonus: track **fasting glucose** over one episode
DART exposes **`fasting_glucose`** in the partial observation (not `patient["glucose"]` from the template). For full trajectories use `GET /state` after each step if you extend the server.

In [ ]:
import random

import matplotlib.pyplot as plt
import requests


def track_fasting_glucose(agent_fn, *, seed=None, max_steps: int = 52):
    seed = seed if seed is not None else random.randint(0, 9999)
    r = requests.post(f"{BASE}/reset", json={"seed": seed}, timeout=60)
    r.raise_for_status()
    data = r.json()
    obs = inner_obs(data)
    series = [float(obs.get("fasting_glucose", 0.0))]
    done = False
    steps = 0
    while not done and steps < max_steps:
        rr = requests.post(f"{BASE}/step", json=agent_fn(obs), timeout=60)
        rr.raise_for_status()
        out = rr.json()
        done = bool(out.get("done", False))
        obs = inner_obs(out)
        series.append(float(obs.get("fasting_glucose", 0.0)))
        steps += 1
    return series


levels = track_fasting_glucose(large_lm_agent, seed=42)
plt.figure(figsize=(7, 3.5))
plt.plot(levels, marker="o", markersize=3)
plt.title("Fasting glucose trajectory (one HTTP episode)")
plt.xlabel("Step (env weeks)")
plt.ylabel("Fasting glucose (mg/dL approx.)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## What to say to judges (accurate)

> We built a **digital twin environment** for sequential diabetes care. Policies act through **structured treatment JSON** each simulated week. Here we compare a **rule-based clinician**, a **small language model**, and a **larger instruction-tuned model** using the **same HTTP environment** and the **same reward rubric** baked into the simulator.
>
> For **learned policies**, we train with **REINFORCE on environment rollouts** and report full metrics — see **`training/DART_Colab_submission.ipynb`** (single link to submit) or `train_reinforce_twin.py`.

**If asked whether the large LM is RL-trained in this notebook:** say no — this section is **online evaluation** of a prompted LM; RL training is the separate script/notebook above.